# Notebook 6 — Toward a Tiny GPT

이제 남은 모듈들을 하나씩 추가하여 **좀 더 GPT다운 구조**로 갑니다.

이번 노트북에서 추가하는 것들:

- multi-head attention
- feedforward network
- residual connection
- layer normalization
- block stacking

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q -O input.txt https://www.gutenberg.org/cache/epub/10/pg10.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 64
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

## 1. Multi-head attention

In [2]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 2. Feedforward + Block

In [3]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 3. Tiny GPT

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 86])


## 4. 학습

In [5]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.5531
epoch  1 | train loss 2.0877
epoch  2 | train loss 1.8617
epoch  3 | train loss 1.7339
epoch  4 | train loss 1.6451
epoch  5 | train loss 1.5868
epoch  6 | train loss 1.5394
epoch  7 | train loss 1.5007
epoch  8 | train loss 1.4767
epoch  9 | train loss 1.4468
epoch 10 | train loss 1.4241
epoch 11 | train loss 1.4098
epoch 12 | train loss 1.3940
epoch 13 | train loss 1.3761
epoch 14 | train loss 1.3655
epoch 15 | train loss 1.3530
epoch 16 | train loss 1.3425
epoch 17 | train loss 1.3325
epoch 18 | train loss 1.3274
epoch 19 | train loss 1.3184
epoch 20 | train loss 1.3102
epoch 21 | train loss 1.3032
epoch 22 | train loss 1.2959
epoch 23 | train loss 1.2885
epoch 24 | train loss 1.2828
epoch 25 | train loss 1.2771
epoch 26 | train loss 1.2729
epoch 27 | train loss 1.2704
epoch 28 | train loss 1.2653
epoch 29 | train loss 1.2594
epoch 30 | train loss 1.2566
epoch 31 | train loss 1.2506
epoch 32 | train loss 1.2494
epoch 33 | train loss 1.2469
epoch 34 | tra

## 5. Sampling

In [6]:
@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device, start_text="1:1", max_new_tokens=400):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_gpt(model, block_size, stoi, itos, device, start_text="1:1", max_new_tokens=500))

1:1 The LORD is from the slain divure and that they out a city possession
to destroy it.

10:29 Since they said unto them, Go, as Hezekiah his son was upon him,
soul that is rent forty to death, to be cut off blind upon me, seeing
and seven thorns, neither shall he shall desire to make known, that
is rulers over indorses the saints of their prey.

33:32 Also he said, Thus saith the LORD; Bless they come to pass, I
because ship with my judgment and righteousness, in painst on God,
thou the noise of 


## 6. 정리

이제 아주 작은 GPT 구조가 완성되었습니다.

전체 흐름은 다음과 같습니다.

1. bigram
2. MLP on names
3. MLP on Shakespeare
4. GPT-style dataset + minimal sequence model
5. single-head masked self-attention
6. tiny GPT

## 7. Beyond notebook_06

In [15]:
@torch.no_grad()
def sample_gpt_temperature(model, block_size, stoi, itos, device,
                           start_text="1:1", max_new_tokens=400, temperature=0.7):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)

    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)

    out = list(start_text)

    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)

    return "".join(out)

print(sample_gpt_temperature(model, block_size, stoi, itos, device, start_text="1:1", max_new_tokens=500, temperature=0.5))

1:13 Then the priests which I am the LORD thy God, and thou shalt not
desire thy sanctuary.

119:29 And when thou shalt testify them against them that which is the
sea, and the famine, and the place of the LORD will I give them unto me
remained to the seventh year of the LORD.

3:14 And it came to pass, when they shall come against the LORD.

2:13 And the LORD said unto Moses, Wherefore they were strong afar
off the house, and because of the congregation.

1:14 And the sons of Aaron and the sons of
